In [ ]:
import sys
sys.path.append("..")

from src import config
from src.file_utils import load_file
from src.factors.utils import get_latest_fundamental_period, get_latest_price_date, datetime_to_string
from src.factors import build_momentum_table, build_quality_table, build_value_table

import pandas as pd

### Load universe

- Use ticker as an index to use `.loc` later

In [2]:
universe = load_file(config.RAW_DATA_DIR / "universe.csv", index_col="Ticker")

In [4]:
fundamental_statement = load_file(config.FUNDAMENTALS_SAVE_DIR / "ABF.L" / "balance_sheet.csv", "columns")
print(fundamental_statement.columns.dtype)

prices = load_file(config.PRICES_SAVE_DIR / "ABF.L.csv", "index")
print(prices.index.dtype)

datetime64[us]
datetime64[us]


In [ ]:
momentum_table = []
quality_table = []
value_table = []

missing_files = []
mismatched_periods = []
missing_dates = []

for ticker in universe.index:
    balance_sheet = load_file(config.FUNDAMENTALS_SAVE_DIR / ticker / "balance_sheet.csv", "columns")
    if balance_sheet is None:
        missing_files.append({"ticker": ticker, "missing_file": "balance_sheet"})

    income_statement = load_file(config.FUNDAMENTALS_SAVE_DIR / ticker / "income_statement.csv", "columns")
    if income_statement is None:
        missing_files.append({"ticker": ticker, "missing_file": "income_statement"})

    cash_flow = load_file(config.FUNDAMENTALS_SAVE_DIR / ticker / "cash_flow.csv", "columns")
    if cash_flow is None:
        missing_files.append({"ticker": ticker, "missing_file": "cash_flow"})

    prices = load_file(config.PRICES_SAVE_DIR / f"{ticker}.csv", "index")
    if prices is None:
        missing_files.append({"ticker": ticker, "missing_file": "prices"})

    metadata = load_file(config.RAW_DATA_DIR / "metadata.csv")
    if metadata is None:
        missing_files.append({"ticker": ticker, "missing_file": "metadata"})

    if any(file is None for file in [balance_sheet, income_statement, cash_flow, prices, metadata]):
        continue

    ticker_country = universe.loc[ticker, "Country"]
    
    for pd_date in config.REBALANCE_DATES:
        string_date = pd_date.strftime("%Y-%m-%d")
        
        latest_balance_sheet_period = get_latest_fundamental_period(balance_sheet, pd_date)
        latest_income_statement_period = get_latest_fundamental_period(income_statement, pd_date)
        latest_cash_flow_period = get_latest_fundamental_period(cash_flow, pd_date)
        latest_price_date = get_latest_price_date(prices, pd_date)

        string_balance_sheet_period = datetime_to_string(latest_balance_sheet_period)
        string_income_statement_period = datetime_to_string(latest_income_statement_period)
        string_cash_flow_period = datetime_to_string(latest_cash_flow_period)
        string_price_date = datetime_to_string(latest_price_date)

        periods = {
            "latest_balance_sheet_period": string_balance_sheet_period, 
            "latest_income_statement_period": string_income_statement_period, 
            "latest_cash_flow_period": string_cash_flow_period
        }

        periods_missing = [k for k, v in periods.items() if v is None]
        if len(periods_missing) > 0:
            missing_dates.append({
                "ticker": ticker,
                "date": string_date,
                "missing_periods": periods_missing,
            })
            continue

        if len(set(periods.values())) != 1:
            mismatched_periods.append({
                "ticker": ticker,
                "date": string_date,
                "balance_sheet_period": string_balance_sheet_period,
                "income_statement_period": string_income_statement_period,
                "cash_flow_period": string_cash_flow_period
            })

        build_quality_table(
            ticker, 
            ticker_country, 
            string_date, 
            balance_sheet, 
            income_statement, 
            latest_balance_sheet_period, 
            latest_income_statement_period, 
            quality_table
        )
        build_value_table(
            ticker, 
            string_date, 
            balance_sheet, 
            cash_flow, 
            income_statement, 
            prices,
            metadata,
            latest_balance_sheet_period,
            latest_income_statement_period,
            latest_cash_flow_period,
            latest_price_date,
            value_table
        )
        build_momentum_table(ticker, string_date, prices, latest_price_date, momentum_table)